In [11]:
import pandas as pd

# Read the parquet file
df_parquet = pd.read_parquet("yellow_tripdata_2025-01.parquet")

# Display the first five rows
df_parquet.head()

df_parquet.sample(5000).to_csv("yellow_tripdata_sample.csv", index=False)

In [12]:
# Import API-related Python modules
import json
import certifi
import urllib3
from urllib3 import request

url = "https://data.cityofnewyork.us/resource/gkne-dk5s.json?$limit=5"

# Initialize HTTPS connection
http = urllib3.PoolManager(cert_reqs='CERT_REQUIRED', ca_certs=certifi.where())

# Make the reuqest
response = http.request('GET', url)

# Check status
print(response.status)

if response.status == 200:
    print("Successful connection")
else:
    print(f"Connection failed. Status code: {response.status}")

data = json.loads(response.data.decode('utf-8'))

print(data[0])




200
Successful connection
{'vendor_id': 'CMT', 'pickup_datetime': '2014-11-23T20:31:29.000', 'dropoff_datetime': '2014-11-23T20:31:29.000', 'passenger_count': '3', 'trip_distance': '0', 'pickup_longitude': '0', 'pickup_latitude': '0', 'store_and_fwd_flag': 'N', 'dropoff_longitude': '0', 'dropoff_latitude': '0', 'payment_type': 'CSH', 'fare_amount': '3', 'mta_tax': '0.5', 'tip_amount': '0', 'tolls_amount': '0', 'total_amount': '4', 'imp_surcharge': '0.5', 'rate_code': '1'}


In [15]:
# Import modules
import json
import sqlite3
import certifi
import pandas as pd
import urllib3
from urllib3 import PoolManager
from bs4 import BeautifulSoup 

# --- CSV ---
def import_csv(filepath:str) -> pd.DataFrame:
    try:
        df = pd.read_csv(filepath)
        print(f"Loaded CSV -> {filepath} ({df.shape[0]} rows)")
        return df
    except Exception as e:
        print(f"CSV import failed: {e}")
        return pd.DataFrame()
    
# --- Parquet ---
def import_parquet(filepath:str) -> pd.DataFrame:
    try:
        df = pd.read_parquet(filepath)
        print(f"Loaded Parquet -> {filepath} {df.shape[0]} rows)")
        return df
    except Exception as e:
        print(f"Parquet import failed: {e}")
        return pd.DataFrame()
    
    # --- API (JSON) ---
    def import_api_json(url:str) -> pd.DataFrame:
        try:
            http = PoolManager(cert_reqs='CERT_REQUIRED', ca_certs=certifi.where())
            response = http.request('GET', url)
            if response.status != 200:
                print(f"API call failed with HTTP status: {response.status}")
                return pd.DataFrame()
            data = json.loads(response.data.decode("utf-8"))
            df = pd.DataFrame(data)
            print(f"Loaded API JSON -> {url} ({df.shape[0]} rows)")
            return df
        except Exception as e:
            print(f"API JSON import failed: {e}")
            return pd.DataFrame
        
        # --- SQLite ---
        def import_sqlite(db_path:str, query:str) -> pd.DataFrame:
            try:
                conn = sqlite3.connect(db_path)
                df = pd.read_sql_query(query, conn)
                conn.close()
                print(f"Loaded SQLite -> '{table_name}' ({df.shape[0]} rows)")
                return df
            except Exception as e:
                print(f"SQLite import filed: {e}")
                return pd.DataFrame()
            
            # --- Web Page (HTML Table) ---
            def import_web_table(url: str) -> pd.DataFrame:
                try:
                    tables = pd.read_html(url)
                    if len(tables) == 0:
                        print(f"No HTML tables found")
                        return pd.DataFrame()
                    df = tables[0]
                    print(f"Loaded HTML table from {url} ({df.shape[0]} rows)")
                    return df
                except Exception as e:
                    print(f"Web import failed: {e}")
                    return pd.DataFrame()
                
            # Universal import wrapper
            def import_all_data(
                    csv_path: str,
                    parquest_path: str,
                    api_url: str, 
                    db_path: str,
                    table_name: str,
                    webpage_url: str,
            ) -> dict:
                """
                Returns a dictionary of DataFrames for all data sources.
                """
                data_sources = {
                    "csv": import_csv(csv_path),
                    "parquet": import_parquet(parquet_path),
                    "api": import_api_json(api_url),
                    "sqlite": import_sqlite(db_path, table_name),
                    "web": import_web_table(webpage_url),
                }
                return data_sources

In [17]:
# Test functions
csv_file = "yellow_tripdata_sample.csv"
parquet_file = "yellow_tripdata_2025-01.parquet"
api_endpoint = "https://data.cityofnewyork.us/resource/gkne-dk5s.json?limit=1000"
sqlite_file = "movies.sqlite"
sqlite_table = "movies"
webpage = "https://www.fdic.gov/resources/resolutions/bank-failures/failed-bank-list/"

data_dict = import_all_data(
    csv_file,
    parquet_file,
    api_endpoint,
    sqlite_file,
    sqlite_table,
    webpage,
)

# Display all shapes
for name, df in data_dict.items():
    print(f"{name.upper()}: -> {df.shape}")


NameError: name 'import_all_data' is not defined